# Spicerack - Recipe Recommender
### DS Club Project - Spring 2026

The idea is pretty simple: you tell us what spices you have, we tell you what you can cook.
Also tells you what spice to buy next to unlock the most new recipes.

Dataset: RecipeNLG (~2.2M recipes from Kaggle)

GitHub: https://github.com/laniel123/DS-Club-Project-Showcase-SpiceRack

---
## Step 1 - Change your inputs here

This is the only cell you need to touch. Edit your pantry and must-use lists, then run all cells.

If you put something in must-use that isnt in your pantry, it gets added automatically.
If a spice isnt recognized it just gets skipped with a warning.

In [37]:
# change these two lists and run everything

# spices you actually have right now
MY_PANTRY = [
    "garlic",
    "cumin",
    "paprika",
    "chili powder",
    "oregano",
    "black pepper",
    "salt",
    "cinnamon",
    "ginger",
]

# spices that MUST show up in every recipe we recommend
# leave as [] if you dont care
MUST_USE = [
    "cumin",
    "garlic",
]

# path to your csv file
CSV_PATH = "cookingdataset/RecipeNLG_dataset.csv"

# how many recipes to load - lower this if its running slow
SAMPLE_SIZE = None  # set to None to load all, or a number like 10000 for a sample

---
## Step 2 - Imports and setup
Run this once, dont touch it

In [38]:
import re
import warnings
import pandas as pd
import numpy as np
import joblib
import time
from sklearn.decomposition import NMF
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import pairwise_distances

# pulling all our spice data from the external file
# spice_data.py needs to be in the same folder as this notebook
from spice_data_v2 import SPICES, ALIASES, CANONICAL_SPICES, FLAVOR_PROFILES, REGION_PROFILES

print(f"loaded {len(CANONICAL_SPICES)} spices, {len(FLAVOR_PROFILES)} flavor profiles, {len(REGION_PROFILES)} regions")

loaded 179 spices, 15 flavor profiles, 16 regions


In [39]:
# helper functions used throughout

from spice_data_v2 import SPICES, ALIASES, CANONICAL_SPICES

def clean(s) -> str:
    # lowercase and strip weird characters
    if not isinstance(s, str):
        return ""
    s = s.lower()
    s = re.sub(r"[^a-z\s']", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def to_canonical(spice):
    # normalize a spice name and apply alias mapping
    n = clean(spice)
    return clean(ALIASES.get(n, n))


def validate_spices(raw_list, label):
    # make sure everything the user typed is actually in our vocabulary
    # warns on anything we dont recognize and just skips it
    valid = set()
    bad = []
    for s in raw_list:
        c = to_canonical(s)
        if c in CANONICAL_SPICES:
            valid.add(c)
        else:
            bad.append(s)
    if bad:
        print(f"warning ({label}): didnt recognize these, skipping: {bad}")
    return valid


# build regex patterns for spice extraction from recipe text
# sorted by length so longer matches beat shorter ones (smoked paprika before paprika)
SPICE_PATTERNS = sorted(
    [(sp, clean(sp), re.compile(rf"(^| ){re.escape(clean(sp))}( |$)")) for sp in SPICES],
    key=lambda x: -len(x[0])
)

def get_spices_from_recipe(ingredients):
    # takes a recipe's ingredient list and returns which spices are in it
    if isinstance(ingredients, list):
        raw = " ".join(str(x) for x in ingredients)
    else:
        raw = str(ingredients)
    text = clean(raw)
    found = set()
    for _, norm, pat in SPICE_PATTERNS:
        if pat.search(" " + text + " "):
            found.add(norm)
    return {to_canonical(sp) for sp in found}


def parse_ingredient_string(x) -> list[str]:
    # ingredients are stored as a stringified python list in the csv
    # like '["1 cup flour", "2 eggs"]' so we parse it back
    if isinstance(x, list):
        return [str(i) for i in x]
    if not isinstance(x, str):
        return []
    s = x.strip()
    if s.startswith("[") and s.endswith("]"):
        items = re.findall(r"'([^']*)'|\"([^\"]*)\"", s)
        parsed = [a if a else b for a, b in items]
        return parsed if parsed else [s]
    return [s]



print("helper functions ready")

helper functions ready


---
## Step 3 - Load the data
Run this once per session. Takes a minute on the full dataset.

In [40]:
df = pd.read_csv(CSV_PATH)

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
df.columns

Index(['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
       'NER'],
      dtype='object')

In [ ]:
df.head()

,Unnamed: 0,title,ingredients,directions,link,source,NER
0,0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu..."
1,1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom..."
2,2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar..."
3,3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroo..."
4,4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""bu..."


In [ ]:
#sample_df = df.sample(n=1000, random_state=42)

#sample_df = df.sample(n=1000, random_state=42)

# save a smaller sample for quick testing without loading the whole thing every time

#sample_df.columns
#sample_df.to_csv('sample_dataset_1000.csv', index=False)



Index(['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source',
       'NER'],
      dtype='object')

In [43]:
print(f"loading {CSV_PATH}...")

df = pd.read_csv(CSV_PATH)

if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

# parse NER column (clean ingredient tokens) instead of raw ingredients
# NER gives ["vanilla", "flour", "butter"] not "1/2 tsp. vanilla extract"
df["ingredients_parsed"] = df["NER"].apply(parse_ingredient_string)
df["spices"] = df["ingredients_parsed"].apply(get_spices_from_recipe)

# binary matrix: rows = recipes, columns = spices, 1 if recipe contains that spice
mlb = MultiLabelBinarizer(classes=sorted(CANONICAL_SPICES))
X = np.asarray(mlb.fit_transform(df["spices"]))

print(f"done - {len(df):,} recipes loaded")
print(f"matrix shape: {X.shape}")
print(f"avg spices per recipe: {X.sum(axis=1).mean():.1f}")

loading cookingdataset/RecipeNLG_dataset.csv...
done - 2,231,142 recipes loaded
matrix shape: (2231142, 179)
avg spices per recipe: 1.7


In [44]:
test_data = df.copy()

df.head()

,Unnamed: 0,title,ingredients,directions,link,source,NER,ingredients_parsed,spices
0,0,No-Bake Nut Cookies,"[""1 c. firmly packed brown sugar"", ""1/2 c. eva...","[""In a heavy 2-quart saucepan, mix brown sugar...",www.cookbooks.com/Recipe-Details.aspx?id=44874,Gathered,"[""brown sugar"", ""milk"", ""vanilla"", ""nuts"", ""bu...","[brown sugar, milk, vanilla, nuts, butter, bit...",{vanilla}
1,1,Jewell Ball'S Chicken,"[""1 small jar chipped beef, cut up"", ""4 boned ...","[""Place chipped beef on bottom of baking dish....",www.cookbooks.com/Recipe-Details.aspx?id=699419,Gathered,"[""beef"", ""chicken breasts"", ""cream of mushroom...","[beef, chicken breasts, cream of mushroom soup...",{}
2,2,Creamy Corn,"[""2 (16 oz.) pkg. frozen corn"", ""1 (8 oz.) pkg...","[""In a slow cooker, combine all ingredients. C...",www.cookbooks.com/Recipe-Details.aspx?id=10570,Gathered,"[""frozen corn"", ""cream cheese"", ""butter"", ""gar...","[frozen corn, cream cheese, butter, garlic pow...","{salt, garlic}"
3,3,Chicken Funny,"[""1 large whole chicken"", ""2 (10 1/2 oz.) cans...","[""Boil and debone chicken."", ""Put bite size pi...",www.cookbooks.com/Recipe-Details.aspx?id=897570,Gathered,"[""chicken"", ""chicken gravy"", ""cream of mushroo...","[chicken, chicken gravy, cream of mushroom sou...",{}
4,4,Reeses Cups(Candy),"[""1 c. peanut butter"", ""3/4 c. graham cracker ...","[""Combine first four ingredients and press in ...",www.cookbooks.com/Recipe-Details.aspx?id=659239,Gathered,"[""peanut butter"", ""graham cracker crumbs"", ""bu...","[peanut butter, graham cracker crumbs, butter,...",{}


In [ ]:
#sample_df.columns

SyntaxError: invalid syntax (3199194828.py, line 3)

In [ ]:
#sample_df.head(15)

NameError: name 'sample_df' is not defined

In [45]:
print(f"df shape: {df.shape}")
print(f"X shape: {X.shape}")
print(f"spices sample: {list(df['spices'].iloc[0])}")
print(f"non-empty spice rows: {df['spices'].apply(len).gt(0).sum()}")
print(f"avg spices per recipe: {df['spices'].apply(len).mean():.2f}")

df shape: (2231142, 9)
X shape: (2231142, 179)
spices sample: ['vanilla']
non-empty spice rows: 1649782
avg spices per recipe: 1.70


In [46]:
print("first 5 spice sets:")
for i in range(5):
    print(f"  {df['spices'].iloc[i]}")

print()
print("first 5 ingredients_parsed:")
for i in range(5):
    print(f"  {df['ingredients_parsed'].iloc[i][:3]}")

print()
print(f"does df have ingredients_parsed col: {'ingredients_parsed' in df.columns}")
print(f"does df have spices col: {'spices' in df.columns}")
print(f"df columns: {list(df.columns)}")

first 5 spice sets:
  {'vanilla'}
  set()
  {'salt', 'garlic'}
  set()
  set()

first 5 ingredients_parsed:
  ['brown sugar', 'milk', 'vanilla']
  ['beef', 'chicken breasts', 'cream of mushroom soup']
  ['frozen corn', 'cream cheese', 'butter']
  ['chicken', 'chicken gravy', 'cream of mushroom soup']
  ['peanut butter', 'graham cracker crumbs', 'butter']

does df have ingredients_parsed col: True
does df have spices col: True
df columns: ['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source', 'NER', 'ingredients_parsed', 'spices']


In [ ]:
from sklearn.cluster       import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics       import silhouette_score

N_CLUSTERS = 100   # bump to 30-50 for full 2M dataset

# ── cluster ───────────────────────────────────────────────────────────────
# X is (n_recipes, 179) binary matrix from cell 11
# MiniBatchKMeans is the same as KMeans but fast enough for 2M rows

kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS, random_state=42,
    batch_size=4096, n_init=10
)
df["cluster"] = kmeans.fit_predict(X)

# 1. initial clustering
kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS, random_state=42,
    batch_size=4096, n_init=10
)
df["cluster"] = kmeans.fit_predict(X)

# 2. split oversized clusters
SIZE_THRESHOLD = 50_000   # any cluster bigger than this gets split
SUBCLUSTERS    = 5         # how many pieces to break it into
next_id        = N_CLUSTERS  # new sub-cluster IDs start here

for cid in range(N_CLUSTERS):
    mask = df["cluster"] == cid
    if mask.sum() <= SIZE_THRESHOLD:
        continue  # fine as-is, skip

    print(f"splitting cluster {cid} ({mask.sum():,} recipes) into {SUBCLUSTERS}...")
    sub_X      = X[mask.values]
    sub_kmeans = MiniBatchKMeans(
        n_clusters=SUBCLUSTERS, random_state=42,
        batch_size=4096, n_init=5
    )
    sub_labels = sub_kmeans.fit_predict(sub_X)

    # assign new IDs to each sub-cluster
    df.loc[mask, "cluster"] = sub_labels + next_id
    next_id += SUBCLUSTERS

print(f"\ntotal clusters after splitting: {df['cluster'].nunique()}")
print(df["cluster"].value_counts().sort_index().to_string())

print(f"cluster sizes:")
print(df["cluster"].value_counts().sort_index().to_string())

# silhouette score on a 5k sample — tells you how good the clusters are
# >0.2 is reasonable, >0.4 is good
idx = np.random.choice(len(X), min(5000, len(X)), replace=False)
sil = silhouette_score(X[idx], df["cluster"].iloc[idx], metric="jaccard")
print(f"\nsilhouette: {sil:.3f}")

In [ ]:
# remove empty clusters from df and remap IDs to be contiguous
df = df[df["cluster"].map(df["cluster"].value_counts()) > 0].reset_index(drop=True)

# remap cluster IDs to 0, 1, 2... with no gaps
old_ids = sorted(df["cluster"].unique())
id_map  = {old: new for new, old in enumerate(old_ids)}
df["cluster"] = df["cluster"].map(id_map)

# also remap cluster_top_spices to match new IDs
cluster_top_spices = {id_map[k]: v for k, v in cluster_top_spices.items() if k in id_map}

print(f"final cluster count: {df['cluster'].nunique()}")
print(f"recipes kept: {len(df):,}")

idx = np.random.choice(len(X), min(5000, len(X)), replace=False)
sil = silhouette_score(X[idx], df["cluster"].iloc[idx], metric="jaccard")
print(f"silhouette after splitting: {sil:.3f}")

final cluster count: 116
recipes kept: 2,231,142


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/metrics/pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


silhouette after splitting: 0.620


In [70]:
# what did each cluster find?
# read the top spices per cluster — these describe the flavor group
spice_cols = list(mlb.classes_)
cluster_top_spices = {}

for cid in range(N_CLUSTERS):
    mask    = df["cluster"] == cid
    freq    = X[mask.values].mean(axis=0)
    top_idx = freq.argsort()[::-1][:8]
    top     = [spice_cols[i] for i in top_idx if freq[i] > 0]
    cluster_top_spices[cid] = top
    print(f"cluster {cid:>2}  ({mask.sum():>5} recipes)  {', '.join(top[:6])}")

cluster  0  (18976 recipes)  black pepper, kosher salt, salt, garlic, mustard, parsley
cluster  1  (21659 recipes)  cilantro, paprika, curry powder, oregano, cayenne, cumin
cluster  2  (11710 recipes)  jalapeno, chili powder, cilantro, cumin, cayenne, paprika
cluster  3  (12919 recipes)  vanilla, salt, cream of tartar, nutmeg, kosher salt, sea salt
cluster  4  ( 3772 recipes)  vanilla, mint, ginger, pumpkin pie spice, nutmeg, cardamom
cluster  5  (15598 recipes)  garlic, salt, mustard, chili powder, cilantro, dill
cluster  6  (28461 recipes)  salt, basil, chili powder, oregano, seasoned salt, sage
cluster  7  (16174 recipes)  basil, garlic, oregano, black pepper, thyme, cilantro
cluster  8  (24079 recipes)  ginger, cinnamon, cilantro, mint, mustard, coriander
cluster  9  (17184 recipes)  chili powder, cumin, garlic, salt, cloves, oregano
cluster 10  (15461 recipes)  garlic, cloves, basil, black pepper, cilantro, mustard
cluster 11  (42667 recipes)  mustard, salt, black pepper, celery s

In [1]:
N_SVD_DIMS = 100   # bump to 50-100 for full dataset

# compress each recipe from 179 binary dims to 30 dense dims
# after normalize, cosine similarity = dot product (fast)
svd         = TruncatedSVD(n_components=N_SVD_DIMS, random_state=42)
recipe_vecs = svd.fit_transform(X)
recipe_vecs = normalize(recipe_vecs, norm="l2")

print(f"recipe matrix: {recipe_vecs.shape}")
print(f"variance explained: {svd.explained_variance_ratio_.sum():.1%}")

# save everything the website needs
model = {
    "kmeans":             kmeans,
    "svd":                svd,
    "mlb":                mlb,
    "recipe_matrix":      recipe_vecs,
    "recipe_titles":      df["title"].tolist(),
    "recipe_spices":      [list(s) for s in df["spices"]],
    "cluster_labels":     df["cluster"].tolist(),
    "cluster_top_spices": cluster_top_spices,
    "n_clusters":         N_CLUSTERS,
    "n_recipes":          len(df),
}
joblib.dump(model, "spicerack_model.joblib", compress=3)
print("saved → spicerack_model.joblib")

NameError: name 'TruncatedSVD' is not defined

---
## Step 4 - Model functions
These dont need to be touched either. Just run them.

In [36]:
def get_flavor_profiles(pantry, min_coverage=0.3):
    # check how well the users spices cover each flavor profile
    # only returns profiles where they have at least min_coverage fraction of the spices
    results = []
    for profile, spice_set in FLAVOR_PROFILES.items():
        matched = pantry & spice_set
        coverage = len(matched) / len(spice_set)
        if coverage >= min_coverage:
            results.append((profile, round(coverage, 2), sorted(matched)))
    return sorted(results, key=lambda x: -x[1])


def get_regions(profile_names):
    # maps matched flavor profiles to culinary regions
    results = []
    for region, profiles in REGION_PROFILES.items():
        matched = [p for p in profiles if p in profile_names]
        if matched:
            results.append((region, matched))
    return sorted(results, key=lambda x: -len(x[1]))


def recommend(pantry, must_use, top_k=10, min_match=2):
    # jaccard similarity - compares users spice set to each recipes spice set
    # must_use filter removes any recipe that doesnt have all the required spices
    user_vec = np.asarray(mlb.transform([pantry]))

    if must_use:
        spice_cols = list(mlb.classes_)
        must_idx = [spice_cols.index(s) for s in must_use if s in spice_cols]
        if must_idx:
            must_mask = X[:, must_idx].min(axis=1).astype(bool)
        else:
            must_mask = np.ones(len(df), dtype=bool)
    else:
        must_mask = np.ones(len(df), dtype=bool)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sims = 1.0 - pairwise_distances(user_vec, X, metric="jaccard").flatten()

    match_counts = (X & user_vec).sum(axis=1)
    valid = np.where(must_mask & (match_counts >= min_match))[0]

    if len(valid) == 0:
        print("no recipes matched the must-use filter, showing best results without it")
        valid = np.where(match_counts >= min_match)[0]

    ranked = valid[np.lexsort((-match_counts[valid], -sims[valid]))][:top_k]

    out = df.loc[ranked, ["title"]].copy()
    out["similarity"]     = sims[ranked].round(3)
    out["matched_spices"] = df.loc[ranked, "spices"].apply(lambda s: sorted(s & pantry))
    out["num_matched"]    = match_counts[ranked]
    return out.sort_values(["similarity", "num_matched"], ascending=False).reset_index(drop=True)


def tag_region(recipe_spices):
    # assign a single region label to a recipe based on its spice set
    profiles = get_flavor_profiles(recipe_spices, min_coverage=0.2)
    if not profiles:
        return "Other"
    top_profile = profiles[0][0]
    for region, region_profiles in REGION_PROFILES.items():
        if top_profile in region_profiles:
            return region
    return "Other"


def suggest_next_spice(pantry, top_k=5, min_match=2, threshold=0.4):
    # for every spice you dont have, simulate adding it
    # count how many new recipes become available and rank by that
    baseline = recommend(pantry, must_use=set(), top_k=10_000, min_match=min_match)
    baseline_titles = set(baseline["title"])
    baseline_count  = len(baseline[baseline["similarity"] >= threshold])

    missing = [s for s in CANONICAL_SPICES if s not in pantry]
    print(f"baseline: {baseline_count} recipes above {threshold} similarity")
    print(f"testing {len(missing)} candidate spices...")

    results = []
    for spice in missing:
        expanded = pantry | {spice}
        new_recs = recommend(expanded, must_use=set(), top_k=10_000, min_match=min_match)
        new_count  = len(new_recs[new_recs["similarity"] >= threshold])
        new_titles = set(new_recs["title"]) - baseline_titles
        results.append({
            "spice":          spice,
            "newly_unlocked": new_count - baseline_count,
            "examples":       list(new_titles)[:3],
        })

    return pd.DataFrame(results).sort_values("newly_unlocked", ascending=False).head(top_k).reset_index(drop=True)


print("model functions ready")

model functions ready


---
## Step 5 - Results
Edit step 1 and re-run just this cell whenever you want to try different spices.

In [37]:
# validate the inputs from step 1
pantry   = validate_spices(MY_PANTRY, label="pantry")
must_use = validate_spices(MUST_USE,  label="must-use")

extras = must_use - pantry
if extras:
    print(f"added to pantry automatically: {extras}")
    pantry |= extras

print(f"\npantry:   {sorted(pantry)}")
print(f"must-use: {sorted(must_use)}")


# --- flavor profiles ---
print("\n" + "-" * 55)
print("flavor profiles")
print("-" * 55)

profiles = get_flavor_profiles(pantry)
if not profiles:
    print("no strong profiles found - try adding more spices")
else:
    for name, score, matched in profiles:
        bar = "#" * int(score * 10) + "." * (10 - int(score * 10))
        print(f"  [{bar}] {int(score*100)}%  {name}")
        print(f"           spices: {', '.join(matched)}")


# --- culinary regions ---
print("\n" + "-" * 55)
print("cuisines you can cook")
print("-" * 55)

regions = get_regions([p[0] for p in profiles])
if not regions:
    print("no strong regional match found")
else:
    for region, matched_profiles in regions:
        print(f"  {region}")
        print(f"    via: {', '.join(matched_profiles)}")


# --- top recipe recommendations ---
print("\n" + "-" * 55)
if must_use:
    print(f"top recipes (must contain: {sorted(must_use)})")
else:
    print("top recipes")
print("-" * 55)

recs = recommend(pantry, must_use, top_k=10, min_match=2)
print(recs[["title", "similarity", "matched_spices", "num_matched"]].to_string(index=False))


# --- recipes grouped by region ---
print("\n" + "-" * 55)
print("recipes by region")
print("-" * 55)

pool = recommend(pantry, must_use, top_k=200, min_match=2)
pool["region"] = pool["title"].apply(
    lambda t: tag_region(
        df[df["title"] == t]["spices"].iloc[0]
        if (df["title"] == t).any() else set()
    )
)

for region in pool["region"].unique():
    if region == "Other":
        continue
    subset = pool[pool["region"] == region].head(3)
    print(f"\n  {region}")
    for _, row in subset.iterrows():
        print(f"    - {row['title']}  (similarity: {row['similarity']})")


# --- what spice should you buy next ---
print("\n" + "-" * 55)
print("spices to buy next")
print("-" * 55)

next_spice = suggest_next_spice(pantry, top_k=5, min_match=2)
for _, row in next_spice.iterrows():
    print(f"  {row['spice']:<25} {row['newly_unlocked']:+} new recipes")
    if row["examples"]:
        print(f"  {'':25} e.g. {row['examples'][0]}") 


pantry:   ['black pepper', 'chili powder', 'cinnamon', 'cumin', 'garlic', 'ginger', 'oregano', 'paprika', 'salt']
must-use: ['cumin', 'garlic']

-------------------------------------------------------
flavor profiles
-------------------------------------------------------
  [###.......] 31%  American BBQ & Southern
           spices: black pepper, chili powder, cumin, garlic, paprika

-------------------------------------------------------
cuisines you can cook
-------------------------------------------------------
  American BBQ / Southern
    via: American BBQ & Southern

-------------------------------------------------------
top recipes (must contain: ['cumin', 'garlic'])
-------------------------------------------------------
                                                    title  similarity                                                      matched_spices  num_matched
                                          Chicken Karrahi       0.500                         [black peppe